# LLM-Equity Training Notebook (FLAG-Trader for Stocks)

**Fase 2 del piano IB Bot Orizzontale**

Questo notebook addestra un modello FLAG-Trader (Qwen 0.5B + PPO) per trading azionario daily.
Adattato dal crypto_bot esistente con:
- TP/SL range equity: TP [1%, 8%], SL [0.5%, 4%]
- Transaction cost 1 bps (IBKR quasi gratis vs crypto 7 bps)
- Observation shape (N, 5) solo OHLCV (no proxy features crypto)
- 252 trading days/anno
- Enhanced Sharpe reward

**Self-contained**: tutto il codice e' inline, non serve il repo.

**Runtime**: Colab T4, circa 3h per ciclo completo.

## 1. Setup & Dependencies

In [ ]:
# Install dependencies
!pip install -q transformers torch gymnasium peft accelerate yfinance tqdm pandas numpy matplotlib

# Optional: clone repo for reference (not needed, everything is inline)
# !git clone https://github.com/fracarlesi/tradercripto.git
# %cd tradercripto

In [ ]:
import os
import math
import json
import logging
import contextlib
from pathlib import Path
from datetime import datetime, timedelta
from typing import Any, Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.distributions import Categorical
import gymnasium as gym
from gymnasium import spaces
from tqdm.auto import tqdm
import yfinance as yf
from transformers import AutoModelForCausalLM, AutoTokenizer

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# Device
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")

## 2. Download Data

Download top 100 S&P 500 stocks via yfinance (2 anni daily).
Stessa logica di `ib_bot/scripts/download_training_data.py`.

In [ ]:
# S&P 500 top 100 by market cap
SP500_TOP100 = [
    "AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "GOOG", "BRK-B",
    "LLY", "AVGO", "JPM", "TSLA", "UNH", "XOM", "V", "MA", "PG",
    "COST", "JNJ", "HD", "ABBV", "WMT", "NFLX", "BAC", "CRM", "ORCL",
    "CVX", "MRK", "KO", "AMD", "PEP", "ADBE", "TMO", "ACN", "LIN",
    "MCD", "CSCO", "ABT", "WFC", "PM", "DHR", "NOW", "QCOM", "GE",
    "TXN", "INTU", "CAT", "ISRG", "VZ", "AMGN", "CMCSA", "IBM",
    "AMAT", "MS", "PFE", "DIS", "GS", "NEE", "AXP", "HON", "UNP",
    "LOW", "RTX", "BKNG", "SYK", "SPGI", "BLK", "UBER", "T", "COP",
    "ELV", "PLD", "MDLZ", "SCHW", "DE", "LRCX", "VRTX", "BMY",
    "PANW", "ADP", "CB", "GILD", "KLAC", "ADI", "SBUX", "TMUS",
    "FI", "SO", "MU", "CI", "BSX", "MMC", "DUK", "SHW", "ICE",
    "CME", "REGN", "PGR", "CL", "SNPS",
]

DATA_DIR = Path("data/training/equity_daily")
DATA_DIR.mkdir(parents=True, exist_ok=True)

DAYS = 730  # 2 years
end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=DAYS)).strftime("%Y-%m-%d")

print(f"Downloading {len(SP500_TOP100)} symbols, period {start_date} to {end_date}")

success = 0
failed = []

for symbol in tqdm(SP500_TOP100, desc="Downloading", unit="symbol"):
    try:
        ticker = yf.Ticker(symbol)
        df = ticker.history(start=start_date, end=end_date, interval="1d", auto_adjust=True)
        if df.empty:
            failed.append(symbol)
            continue
        df = df.reset_index()
        df = df.rename(columns={
            "Date": "date", "Open": "open", "High": "high",
            "Low": "low", "Close": "close", "Volume": "volume",
        })
        df = df[["date", "open", "high", "low", "close", "volume"]]
        df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None).dt.normalize()
        df = df.dropna(subset=["open", "high", "low", "close"])
        if len(df) < 10:
            failed.append(symbol)
            continue
        df.to_parquet(DATA_DIR / f"{symbol}.parquet", index=False)
        success += 1
    except Exception as e:
        logger.error(f"Error downloading {symbol}: {e}")
        failed.append(symbol)

print(f"\nDownload complete: {success}/{len(SP500_TOP100)} successful")
if failed:
    print(f"Failed ({len(failed)}): {', '.join(failed)}")

## 3. Prepare Training Episodes

Calcola indicatori tecnici e trova setup (squeeze, EMA crossover).
Stessa logica di `ib_bot/scripts/prepare_training_data.py`.

In [ ]:
# ============================================================
# Indicator calculations (pure numpy)
# From: ib_bot/scripts/prepare_training_data.py
# ============================================================

def ind_ema(series: np.ndarray, period: int) -> np.ndarray:
    """Exponential moving average."""
    alpha = 2.0 / (period + 1)
    result = np.empty_like(series, dtype=np.float64)
    result[0] = series[0]
    for i in range(1, len(series)):
        result[i] = alpha * series[i] + (1 - alpha) * result[i - 1]
    return result


def ind_sma(series: np.ndarray, period: int) -> np.ndarray:
    """Simple moving average with NaN fill for initial period."""
    result = np.full_like(series, np.nan, dtype=np.float64)
    if len(series) < period:
        return result
    cumsum = np.cumsum(series)
    result[period - 1:] = (
        cumsum[period - 1:] - np.concatenate([[0], cumsum[:-period]])
    ) / period
    return result


def ind_rsi(close: np.ndarray, period: int = 14) -> np.ndarray:
    """Relative Strength Index."""
    result = np.full_like(close, np.nan, dtype=np.float64)
    if len(close) < period + 1:
        return result
    delta = np.diff(close)
    gains = np.where(delta > 0, delta, 0.0)
    losses = np.where(delta < 0, -delta, 0.0)
    avg_gain = np.mean(gains[:period])
    avg_loss = np.mean(losses[:period])
    for i in range(period, len(delta)):
        avg_gain = (avg_gain * (period - 1) + gains[i]) / period
        avg_loss = (avg_loss * (period - 1) + losses[i]) / period
        if avg_loss == 0:
            result[i + 1] = 100.0
        else:
            rs = avg_gain / avg_loss
            result[i + 1] = 100.0 - (100.0 / (1.0 + rs))
    if avg_loss == 0:
        result[period] = 100.0
    else:
        first_gain = np.mean(gains[:period])
        first_loss = np.mean(losses[:period])
        if first_loss == 0:
            result[period] = 100.0
        else:
            result[period] = 100.0 - (100.0 / (1.0 + first_gain / first_loss))
    return result


def ind_atr(high: np.ndarray, low: np.ndarray, close: np.ndarray, period: int = 14) -> np.ndarray:
    """Average True Range."""
    result = np.full(len(high), np.nan, dtype=np.float64)
    if len(high) < period + 1:
        return result
    tr = np.empty(len(high), dtype=np.float64)
    tr[0] = high[0] - low[0]
    for i in range(1, len(high)):
        tr[i] = max(high[i] - low[i], abs(high[i] - close[i - 1]), abs(low[i] - close[i - 1]))
    result[period] = np.mean(tr[1:period + 1])
    for i in range(period + 1, len(high)):
        result[i] = (result[i - 1] * (period - 1) + tr[i]) / period
    return result


def detect_squeeze(
    close: np.ndarray, high: np.ndarray, low: np.ndarray,
    bb_period: int = 20, bb_mult: float = 2.0,
    kc_period: int = 20, kc_mult: float = 1.5,
) -> np.ndarray:
    """Detect squeeze states. 0=no, 1=on, 2=fired."""
    n = len(close)
    result = np.zeros(n, dtype=np.int8)
    if n < max(bb_period, kc_period) + 1:
        return result
    bb_sma_arr = ind_sma(close, bb_period)
    bb_std = np.full(n, np.nan, dtype=np.float64)
    for i in range(bb_period - 1, n):
        bb_std[i] = np.std(close[i - bb_period + 1:i + 1], ddof=0)
    bb_upper = bb_sma_arr + bb_mult * bb_std
    bb_lower = bb_sma_arr - bb_mult * bb_std
    kc_ema_arr = ind_ema(close, kc_period)
    atr_vals = ind_atr(high, low, close, kc_period)
    kc_upper = kc_ema_arr + kc_mult * atr_vals
    kc_lower = kc_ema_arr - kc_mult * atr_vals
    squeeze_on = np.zeros(n, dtype=bool)
    for i in range(max(bb_period, kc_period), n):
        if not np.isnan(bb_lower[i]) and not np.isnan(kc_lower[i]):
            squeeze_on[i] = (bb_lower[i] > kc_lower[i]) and (bb_upper[i] < kc_upper[i])
    for i in range(1, n):
        if squeeze_on[i]:
            result[i] = 1
        elif squeeze_on[i - 1] and not squeeze_on[i]:
            result[i] = 2
    return result


def detect_setups(df: pd.DataFrame) -> pd.DataFrame:
    """Detect trading setups and compute forward returns."""
    close = df["close"].values.astype(np.float64)
    high = df["high"].values.astype(np.float64)
    low = df["low"].values.astype(np.float64)
    n = len(close)
    if n < 60:
        return pd.DataFrame()
    ema4 = ind_ema(close, 4)
    ema21 = ind_ema(close, 21)
    sma50 = ind_sma(close, 50)
    rsi14 = ind_rsi(close, 14)
    atr14 = ind_atr(high, low, close, 14)
    squeeze = detect_squeeze(close, high, low)
    setups = []
    for i in range(1, n):
        entry_price = close[i]
        if entry_price <= 0 or np.isnan(entry_price):
            continue
        fwd = {}
        for days, label in [(1, "result_1d"), (3, "result_3d"), (5, "result_5d"), (8, "result_8d")]:
            if i + days < n:
                fwd[label] = ((close[i + days] - entry_price) / entry_price) * 100
            else:
                fwd[label] = np.nan
        base = {
            "date": df["date"].iloc[i], "entry_price": entry_price,
            "ema4": ema4[i], "ema21": ema21[i], "sma50": sma50[i],
            "rsi14": rsi14[i], "atr14": atr14[i], **fwd,
        }
        if squeeze[i] == 2:
            direction = "long" if ema4[i] > ema21[i] else "short"
            setups.append({**base, "signal_type": f"squeeze_{direction}"})
        if i >= 2 and ema4[i - 1] <= ema21[i - 1] and ema4[i] > ema21[i]:
            setups.append({**base, "signal_type": "long"})
        if i >= 2 and ema4[i - 1] >= ema21[i - 1] and ema4[i] < ema21[i]:
            setups.append({**base, "signal_type": "short"})
    return pd.DataFrame(setups) if setups else pd.DataFrame()

In [ ]:
# Process all downloaded symbols
parquet_files = sorted(DATA_DIR.glob("*.parquet"))
print(f"Processing {len(parquet_files)} symbol files")

all_episodes = []
all_candle_data = {}  # symbol -> DataFrame (for env later)

for pf in tqdm(parquet_files, desc="Processing", unit="symbol"):
    symbol = pf.stem
    try:
        df = pd.read_parquet(pf)
        df = df.sort_values("date").reset_index(drop=True)
        all_candle_data[symbol] = df
        episodes = detect_setups(df)
        if not episodes.empty:
            episodes.insert(0, "symbol", symbol)
            all_episodes.append(episodes)
    except Exception as e:
        logger.error(f"Error processing {symbol}: {e}")

if all_episodes:
    episodes_df = pd.concat(all_episodes, ignore_index=True)
    episodes_df.to_parquet("data/training/equity_episodes.parquet", index=False)
    print(f"\nTotal episodes: {len(episodes_df)}")
    print(f"Symbols with setups: {episodes_df['symbol'].nunique()}")
    print(f"\nSignal distribution:")
    print(episodes_df["signal_type"].value_counts())
else:
    print("ERROR: No episodes generated")

## 4. Equity Trading Environment

Gymnasium env per azioni, adattato da `crypto_bot/flag_trader/environment.py`.

Differenze dal crypto env:
- `transaction_cost_bps = 1.0` (IBKR quasi gratis vs crypto 5-7 bps)
- Observation shape `(N, 5)` solo OHLCV (no proxy features crypto: funding, OI, vol_delta)
- 252 trading days/anno
- Reward: enhanced_sharpe (riuso da crypto)

In [ ]:
# ============================================================
# Reward functions (from crypto_bot/flag_trader/reward.py)
# ============================================================

def compute_sharpe_delta(
    pnl_history: list[float],
    risk_free_rate: float = 0.0,
    **kwargs: Any,
) -> float:
    """Incremental Sharpe ratio change: SR_t - SR_{t-1}."""
    if len(pnl_history) < 2:
        return 0.0

    def _sharpe(values: list[float]) -> float:
        n = len(values)
        if n < 2:
            return 0.0
        mean = sum(values) / n
        variance = sum((v - mean) ** 2 for v in values) / (n - 1)
        std = math.sqrt(variance)
        if std < 1e-12:
            return 0.0
        return (mean - risk_free_rate) / std

    sr_prev = _sharpe(pnl_history[:-1])
    sr_curr = _sharpe(pnl_history)
    return sr_curr - sr_prev


def compute_enhanced_sharpe(
    pnl_history: list[float],
    risk_free_rate: float = 0.0,
    trade_opened: bool = False,
    trade_closed_profit: bool | None = None,
    **kwargs: Any,
) -> float:
    """Enhanced Sharpe delta con bonus/penalita per segnale piu chiaro.

    1. Base: sharpe_delta
    2. PnL bonus: +0.1 se trade chiuso in profitto, -0.1 se in loss
    3. Fee penalty: -0.01 se trade aperto (scoraggia overtrading)
    """
    reward = compute_sharpe_delta(pnl_history, risk_free_rate)
    if trade_closed_profit is True:
        reward += 0.1
    elif trade_closed_profit is False:
        reward -= 0.1
    if trade_opened:
        reward -= 0.01
    return reward

In [ ]:
# ============================================================
# Equity Trading Environment
# Adapted from: crypto_bot/flag_trader/environment.py
# Key differences:
#   - transaction_cost_bps = 1.0 (IBKR vs crypto 5-7 bps)
#   - obs shape (N, 5) OHLCV only (no proxy features)
#   - enhanced_sharpe reward by default
# ============================================================

class EquityTradingEnv(gym.Env):
    """Gymnasium environment for simulated equity trading.

    Args:
        candles: np.ndarray of shape (num_candles, 5) -- O, H, L, C, V.
        initial_cash: Starting cash balance.
        transaction_cost_bps: Transaction cost in basis points (1 = 0.01% for IBKR).
        window_size: Number of historical candles in the observation.
    """

    metadata = {"render_modes": []}

    def __init__(
        self,
        candles: np.ndarray,
        initial_cash: float = 10_000.0,
        transaction_cost_bps: float = 1.0,
        window_size: int = 20,
        reward_fn: Callable[..., float] | None = None,
        symbol: str = "",
    ) -> None:
        super().__init__()
        assert candles.ndim == 2 and candles.shape[1] == 5, (
            "candles must be shape (N, 5) with columns [O, H, L, C, V]"
        )
        assert candles.shape[0] > window_size, (
            f"Need more candles ({candles.shape[0]}) than window_size ({window_size})"
        )

        self.candles = candles.astype(np.float32)
        self.initial_cash = initial_cash
        self.tx_cost_pct = transaction_cost_bps / 10_000.0
        self.window_size = window_size
        self.reward_history_len = 10
        self._reward_fn: Callable[..., float] = reward_fn or compute_enhanced_sharpe
        self.symbol = symbol

        # Action: 0=Sell, 1=Hold, 2=Buy
        self.action_space = spaces.Discrete(3)

        # Observation: candles (N,5) + portfolio (4,) + history (10,)
        self.observation_space = spaces.Dict({
            "candles": spaces.Box(low=-np.inf, high=np.inf, shape=(window_size, 5), dtype=np.float32),
            "portfolio": spaces.Box(low=-np.inf, high=np.inf, shape=(4,), dtype=np.float32),
            "history": spaces.Box(low=-np.inf, high=np.inf, shape=(self.reward_history_len,), dtype=np.float32),
        })

        # Internal state
        self._step_idx: int = 0
        self._cash: float = 0.0
        self._position: float = 0.0  # shares held
        self._entry_price: float = 0.0
        self._pnl_history: list[float] = []
        self._reward_history: list[float] = []
        self._total_value_prev: float = 0.0

        # Dynamic TP/SL from model (equity ranges)
        self._tp_pct: float = 4.0   # default TP %
        self._sl_pct: float = 2.0   # default SL %
        self._tp_price: float = 0.0
        self._sl_price: float = 0.0

    def reset(self, seed: int | None = None, options: dict[str, Any] | None = None):
        super().reset(seed=seed)
        self._step_idx = self.window_size
        self._cash = self.initial_cash
        self._position = 0.0
        self._entry_price = 0.0
        self._pnl_history = []
        self._reward_history = []
        self._total_value_prev = self.initial_cash
        self._tp_pct = 4.0
        self._sl_pct = 2.0
        self._tp_price = 0.0
        self._sl_price = 0.0
        return self._get_obs(), {"total_value": self.initial_cash, "step": 0}

    def set_tp_sl(self, tp_pct: float, sl_pct: float) -> None:
        """Set TP/SL for next trade. Called before step()."""
        self._tp_pct = tp_pct
        self._sl_pct = sl_pct

    def step(self, action: int):
        current_close = float(self.candles[self._step_idx, 3])
        current_high = float(self.candles[self._step_idx, 1])
        current_low = float(self.candles[self._step_idx, 2])
        executed_action = action

        # Check TP/SL hit
        step_pnl = 0.0
        if self._position > 0:
            if self._tp_price > 0 and current_high >= self._tp_price:
                gross = self._position * self._tp_price
                cost = gross * self.tx_cost_pct
                proceeds = gross - cost
                step_pnl = proceeds - (self._position * self._entry_price)
                self._cash = proceeds
                self._position = 0.0
                self._entry_price = 0.0
                self._tp_price = 0.0
                self._sl_price = 0.0
                executed_action = 1
                return self._finalize_step(step_pnl, executed_action, current_close)
            elif self._sl_price > 0 and current_low <= self._sl_price:
                gross = self._position * self._sl_price
                cost = gross * self.tx_cost_pct
                proceeds = gross - cost
                step_pnl = proceeds - (self._position * self._entry_price)
                self._cash = proceeds
                self._position = 0.0
                self._entry_price = 0.0
                self._tp_price = 0.0
                self._sl_price = 0.0
                executed_action = 1
                return self._finalize_step(step_pnl, executed_action, current_close)

        # Action masking
        if action == 2 and self._position > 0:
            executed_action = 1
        elif action == 0 and self._position <= 0:
            executed_action = 1

        # Execute trade
        step_pnl = 0.0
        if executed_action == 2:  # Buy
            cost = self._cash * self.tx_cost_pct
            investable = self._cash - cost
            self._position = investable / current_close
            self._entry_price = current_close
            self._cash = 0.0
            self._tp_price = current_close * (1 + self._tp_pct / 100)
            self._sl_price = current_close * (1 - self._sl_pct / 100)
        elif executed_action == 0:  # Sell
            gross = self._position * current_close
            cost = gross * self.tx_cost_pct
            proceeds = gross - cost
            step_pnl = proceeds - (self._position * self._entry_price)
            self._cash = proceeds
            self._position = 0.0
            self._entry_price = 0.0
            self._tp_price = 0.0
            self._sl_price = 0.0

        return self._finalize_step(step_pnl, executed_action, current_close)

    def _finalize_step(self, step_pnl: float, executed_action: int, current_close: float):
        unrealized_pnl = 0.0
        position_value = 0.0
        if self._position > 0:
            position_value = self._position * current_close
            unrealized_pnl = position_value - (self._position * self._entry_price)
        total_value = self._cash + position_value
        mtm_pnl = total_value - self._total_value_prev
        self._pnl_history.append(mtm_pnl)
        self._total_value_prev = total_value

        trade_opened = executed_action == 2
        trade_closed_profit: bool | None = None
        if step_pnl != 0.0:
            trade_closed_profit = step_pnl > 0

        try:
            reward = self._reward_fn(
                self._pnl_history,
                trade_opened=trade_opened,
                trade_closed_profit=trade_closed_profit,
            )
        except TypeError:
            reward = self._reward_fn(self._pnl_history)
        self._reward_history.append(reward)

        self._step_idx += 1
        terminated = self._step_idx >= len(self.candles)
        truncated = False
        obs = self._get_obs() if not terminated else self._get_terminal_obs()
        info = {
            "total_value": total_value, "cash": self._cash,
            "position": self._position, "unrealized_pnl": unrealized_pnl,
            "step_pnl": step_pnl, "mtm_pnl": mtm_pnl,
            "executed_action": executed_action, "step": self._step_idx - self.window_size,
        }
        return obs, float(reward), terminated, truncated, info

    def _get_obs(self) -> dict[str, np.ndarray]:
        """Build observation: normalized OHLCV (N,5) + portfolio (4,) + history (10,)."""
        raw = self.candles[self._step_idx - self.window_size:self._step_idx].copy()
        base_close = raw[0, 3]
        if base_close > 0:
            raw[:, :4] = raw[:, :4] / base_close - 1.0  # Normalize OHLC
            raw[:, 4] = raw[:, 4] / (raw[:, 4].mean() + 1e-12)  # Normalize volume

        # Portfolio state
        current_close = float(self.candles[min(self._step_idx, len(self.candles) - 1), 3])
        position_value = self._position * current_close
        unrealized_pnl = position_value - (self._position * self._entry_price) if self._position > 0 else 0.0
        total_value = self._cash + position_value
        normalizer = max(self.initial_cash, 1e-12)
        portfolio = np.array([
            self._cash / normalizer,
            position_value / normalizer,
            unrealized_pnl / normalizer,
            total_value / normalizer,
        ], dtype=np.float32)

        rh = self._reward_history[-self.reward_history_len:]
        padded = [0.0] * (self.reward_history_len - len(rh)) + rh
        history = np.array(padded, dtype=np.float32)

        return {"candles": raw, "portfolio": portfolio, "history": history}

    def _get_terminal_obs(self) -> dict[str, np.ndarray]:
        return {
            "candles": np.zeros((self.window_size, 5), dtype=np.float32),
            "portfolio": np.zeros(4, dtype=np.float32),
            "history": np.zeros(self.reward_history_len, dtype=np.float32),
        }

## 5. Model Setup

FlagTraderModel con Qwen 0.5B, adattato da `crypto_bot/flag_trader/model.py`.

- Range TP/SL equity: TP [1%, 8%], SL [0.5%, 4%]
- 80% frozen, top 20% + 4 heads trainabili
- Policy head (3 actions: Sell/Hold/Buy) + Value head + TP head + SL head

In [ ]:
# ============================================================
# FlagTraderModel for Equity
# Adapted from: crypto_bot/flag_trader/model.py
# Differences:
#   - TP range [1%, 8%] (vs crypto [0.5%, 5%])
#   - SL range [0.5%, 4%] (vs crypto [0.3%, 2%])
# ============================================================

class FlagTraderModel(nn.Module):
    """HuggingFace causal LM with policy and value heads for RL equity trading."""

    NUM_ACTIONS = 3  # Sell=0, Hold=1, Buy=2

    def __init__(
        self,
        model_name: str = "Qwen/Qwen2.5-0.5B-Instruct",
        freeze_pct: float = 0.8,
        device: torch.device = DEVICE,
    ) -> None:
        super().__init__()
        self.device = device
        self.model_name = model_name

        # bfloat16 on CUDA, float32 on MPS/CPU
        load_dtype = torch.bfloat16 if device == torch.device("cuda") else torch.float32

        self.llm = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=load_dtype, trust_remote_code=True,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        hidden_size: int = self.llm.config.hidden_size
        self._freeze_layers(freeze_pct)

        intermediate = 256 if hidden_size >= 1024 else 64

        # Policy head: Sell/Hold/Buy
        self.policy_head = nn.Sequential(
            nn.Linear(hidden_size, intermediate), nn.ReLU(),
            nn.Linear(intermediate, self.NUM_ACTIONS),
        )
        # Value head: scalar state value
        self.value_head = nn.Sequential(
            nn.Linear(hidden_size, intermediate), nn.ReLU(),
            nn.Linear(intermediate, 1),
        )
        # TP head: take-profit [1%, 8%] (equity range)
        self.tp_head = nn.Sequential(
            nn.Linear(hidden_size, intermediate), nn.ReLU(),
            nn.Linear(intermediate, 1), nn.Sigmoid(),
        )
        # SL head: stop-loss [0.5%, 4%] (equity range)
        self.sl_head = nn.Sequential(
            nn.Linear(hidden_size, intermediate), nn.ReLU(),
            nn.Linear(intermediate, 1), nn.Sigmoid(),
        )

        # Equity TP/SL ranges (wider than crypto)
        self.TP_MIN, self.TP_MAX = 1.0, 8.0
        self.SL_MIN, self.SL_MAX = 0.5, 4.0

        self.to(self.device)

    def _get_transformer_layers(self):
        if hasattr(self.llm, "model") and hasattr(self.llm.model, "layers"):
            return self.llm.model.layers
        elif hasattr(self.llm, "transformer") and hasattr(self.llm.transformer, "h"):
            return self.llm.transformer.h
        elif hasattr(self.llm, "model") and hasattr(self.llm.model, "decoder") and hasattr(self.llm.model.decoder, "layers"):
            return self.llm.model.decoder.layers
        return []

    def _get_embeddings(self):
        if hasattr(self.llm, "model") and hasattr(self.llm.model, "embed_tokens"):
            return self.llm.model.embed_tokens
        elif hasattr(self.llm, "transformer") and hasattr(self.llm.transformer, "wte"):
            return self.llm.transformer.wte
        return None

    def _get_inner_model(self):
        if hasattr(self.llm, "model"):
            return self.llm.model
        elif hasattr(self.llm, "transformer"):
            return self.llm.transformer
        return self.llm

    def _freeze_layers(self, freeze_pct: float) -> None:
        embeddings = self._get_embeddings()
        if embeddings is not None:
            for param in embeddings.parameters():
                param.requires_grad = False
        layers = self._get_transformer_layers()
        num_layers = len(layers)
        if num_layers > 0:
            num_frozen = int(num_layers * freeze_pct)
            for i, layer in enumerate(layers):
                if i < num_frozen:
                    for param in layer.parameters():
                        param.requires_grad = False
        else:
            all_params = list(self.llm.parameters())
            for param in all_params:
                param.requires_grad = False
            num_unfreeze = max(1, int(len(all_params) * (1 - freeze_pct)))
            for param in all_params[-num_unfreeze:]:
                param.requires_grad = True
        if hasattr(self.llm, "lm_head"):
            for param in self.llm.lm_head.parameters():
                param.requires_grad = False

    def _extract_last_hidden(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        inner_model = self._get_inner_model()
        outputs = inner_model(
            input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True,
        )
        last_hidden = outputs.last_hidden_state
        seq_lengths = attention_mask.sum(dim=1) - 1
        batch_idx = torch.arange(last_hidden.size(0), device=last_hidden.device)
        hidden = last_hidden[batch_idx, seq_lengths]
        return hidden

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        hidden = self._extract_last_hidden(input_ids, attention_mask)
        hidden = hidden.float()
        logits = self.policy_head(hidden)
        value = self.value_head(hidden)
        tp_raw = self.tp_head(hidden)
        sl_raw = self.sl_head(hidden)
        tp_pct = tp_raw * (self.TP_MAX - self.TP_MIN) + self.TP_MIN
        sl_pct = sl_raw * (self.SL_MAX - self.SL_MIN) + self.SL_MIN
        return logits, value, tp_pct, sl_pct

    @torch.no_grad()
    def get_action(self, prompt: str, return_tokens: bool = False):
        tokens = self.tokenizer(
            prompt, return_tensors="pt", max_length=512, truncation=True, padding=True,
        )
        input_ids = tokens["input_ids"].to(self.device)
        attention_mask = tokens["attention_mask"].to(self.device)
        logits, value, tp_pct_t, sl_pct_t = self.forward(input_ids, attention_mask)
        dist = Categorical(logits=logits.squeeze(0))
        action = dist.sample()
        log_prob = dist.log_prob(action)
        tp = float(tp_pct_t.item())
        sl = float(sl_pct_t.item())
        if return_tokens:
            return int(action.item()), float(value.item()), log_prob, tp, sl, input_ids, attention_mask
        return int(action.item()), float(value.item()), log_prob, tp, sl

    def run_assessment(self, input_ids, attention_mask, actions):
        """Run policy assessment: log_probs, values, entropy, TP/SL for taken actions.
        Used during PPO update step. Runs with gradients enabled.
        """
        logits, values, tp_pct, sl_pct = self.forward(input_ids, attention_mask)
        dist = Categorical(logits=logits)
        log_probs = dist.log_prob(actions)
        entropy = dist.entropy()
        return log_probs, values.squeeze(-1), entropy, tp_pct.squeeze(-1), sl_pct.squeeze(-1)

    def get_trainable_params(self) -> list[nn.Parameter]:
        return [p for p in self.parameters() if p.requires_grad]

    def save_trainable(self, path: Path) -> None:
        trainable_llm = {k: v for k, v in self.llm.named_parameters() if v.requires_grad}
        state = {
            "trainable_layers": {k: v.data for k, v in trainable_llm.items()},
            "policy_head": self.policy_head.state_dict(),
            "value_head": self.value_head.state_dict(),
            "tp_head": self.tp_head.state_dict(),
            "sl_head": self.sl_head.state_dict(),
            "model_name": self.model_name,
            "tp_range": (self.TP_MIN, self.TP_MAX),
            "sl_range": (self.SL_MIN, self.SL_MAX),
        }
        torch.save(state, path)

    def load_trainable(self, path: Path) -> None:
        state = torch.load(path, map_location=self.device, weights_only=True)
        llm_params = dict(self.llm.named_parameters())
        for name, tensor in state["trainable_layers"].items():
            if name in llm_params:
                llm_params[name].data.copy_(tensor)
        self.policy_head.load_state_dict(state["policy_head"])
        self.value_head.load_state_dict(state["value_head"])
        if "tp_head" in state:
            self.tp_head.load_state_dict(state["tp_head"])
        if "sl_head" in state:
            self.sl_head.load_state_dict(state["sl_head"])

In [ ]:
# ============================================================
# Equity Prompt Builder
# Adapted from: crypto_bot/flag_trader/prompt.py
# Differences:
#   - "US stock trading agent" (not crypto)
#   - IBKR fee structure (almost free)
#   - No proxy features (funding, OI, vol_delta)
# ============================================================

class EquityPromptBuilder:
    """Builds structured prompts for the LLM equity trading agent."""

    ACTION_MAP = {"sell": 0, "hold": 1, "buy": 2}

    def __init__(self, candle_window: int = 20, decimal_places: int = 4) -> None:
        self.candle_window = candle_window
        self.decimal_places = decimal_places

    def build_prompt(
        self,
        candles: list[dict[str, float]],
        portfolio: dict[str, float],
        history: dict[str, list],
        market_context: dict | list[dict] | None = None,
    ) -> str:
        trimmed = candles[-self.candle_window:]
        normalized = self._normalize_candles(trimmed)

        prompt = (
            "Task: You are a US stock trading agent. Your goal is to maximize "
            "long-term risk-adjusted returns (grow account equity after fees). "
            "Choose optimal buy, sell, or hold decisions based on market conditions "
            "and risk assessment.\n\n"
            "Trading Costs:\n"
            "- IBKR commission: ~$0.005/share (~0.01% of notional)\n"
            "- Round-trip cost: negligible compared to expected move\n"
            "- Focus on trade quality over cost avoidance\n\n"
            "Legible Actions: Choose from {Buy, Sell, Hold}\n\n"
        )

        if market_context:
            prompt += "Market Context:\n"
            items = market_context if isinstance(market_context, list) else [market_context]
            for ctx in items:
                s = ctx.get("symbol", "?")
                prompt += f"  {s}: "
                parts = []
                for key, label in [("pct_5d", "5d"), ("pct_20d", "20d"), ("vol_ratio", "vol_ratio"), ("rsi_14", "RSI(14)")]:
                    if key in ctx:
                        parts.append(f"{label}: {ctx[key]:.1f}")
                prompt += " | ".join(parts) + "\n"
            prompt += "\n"

        prompt += "Current State:\n"
        state = {
            "historical_prices": normalized,
            "account_status": {
                "cash_balance": round(portfolio.get("cash_balance", 0.0), self.decimal_places),
                "asset_position": round(portfolio.get("asset_position", 0.0), self.decimal_places),
                "total_account_value": round(portfolio.get("total_account_value", 0.0), self.decimal_places),
            },
            "previous_decision_metrics": {
                "recent_rewards": [round(r, self.decimal_places) for r in (history.get("recent_rewards") or [])[-10:]],
                "net_values": [round(v, self.decimal_places) for v in (history.get("net_values") or [])[-10:]],
                "actions": (history.get("actions") or [])[-10:],
            },
        }
        prompt += json.dumps(state, indent=2)
        prompt += (
            '\n\nOutput Action: Format your answer as JSON: '
            '{"Action": "Buy"}, {"Action": "Sell"}, or {"Action": "Hold"}'
        )
        return prompt

    def _normalize_candles(self, candles: list[dict[str, float]]) -> list[dict[str, float]]:
        if not candles:
            return []
        base_close = candles[0].get("close", 1.0)
        if base_close == 0:
            base_close = 1.0
        dp = self.decimal_places
        normalized = []
        for c in candles:
            normalized.append({
                "open": round(c["open"] / base_close - 1.0, dp),
                "high": round(c["high"] / base_close - 1.0, dp),
                "low": round(c["low"] / base_close - 1.0, dp),
                "close": round(c["close"] / base_close - 1.0, dp),
                "volume": round(c["volume"], dp),
            })
        return normalized

In [ ]:
# Initialize model
print("Loading Qwen 0.5B model...")
model = FlagTraderModel(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    freeze_pct=0.8,
    device=DEVICE,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Total params: {total:,}")
print(f"Trainable params: {trainable:,} ({100*trainable/total:.1f}%)")
print(f"TP range: [{model.TP_MIN}%, {model.TP_MAX}%]")
print(f"SL range: [{model.SL_MIN}%, {model.SL_MAX}%]")

## 6. PPO Training

Training loop PPO, adattato da `crypto_bot/flag_trader/trainer.py`.

- lr=3e-5, batch=4, ppo_epochs=4, clip=0.2
- 500 updates x 250 steps
- Multi-asset: rotazione su tutti i simboli
- Salva checkpoint ogni 50 updates

In [ ]:
# ============================================================
# Obs-to-prompt converter for equity
# ============================================================

def obs_to_prompt_inputs_equity(
    obs: dict[str, np.ndarray],
) -> tuple[list[dict[str, float]], dict[str, float], dict[str, list]]:
    """Convert env observation to EquityPromptBuilder format.
    Candles are (N, 5) OHLCV only (no proxy features).
    """
    candles_arr = obs["candles"]  # (window_size, 5)
    candles_list = [
        {
            "open": float(row[0]),
            "high": float(row[1]),
            "low": float(row[2]),
            "close": float(row[3]),
            "volume": float(row[4]),
        }
        for row in candles_arr
    ]
    portfolio_arr = obs["portfolio"]
    portfolio_dict = {
        "cash_balance": float(portfolio_arr[0]),
        "asset_position": float(portfolio_arr[1]),
        "total_account_value": float(portfolio_arr[3]),
    }
    history_arr = obs["history"]
    history_dict = {
        "recent_rewards": [float(x) for x in history_arr],
        "net_values": [],
        "actions": [],
    }
    return candles_list, portfolio_dict, history_dict

In [ ]:
# ============================================================
# RolloutBuffer & PPOTrainer
# Adapted from: crypto_bot/flag_trader/trainer.py
# ============================================================

class RolloutBuffer:
    """Stores transitions collected during rollout for PPO update."""

    def __init__(self) -> None:
        self.states: list[str] = []
        self.actions: list[int] = []
        self.rewards: list[float] = []
        self.values: list[float] = []
        self.log_probs: list[torch.Tensor] = []
        self.dones: list[bool] = []
        self.input_ids: list[torch.Tensor] = []
        self.attention_masks: list[torch.Tensor] = []
        self.tp_pcts: list[float] = []
        self.sl_pcts: list[float] = []

    def add(self, state, action, reward, value, log_prob, done, input_ids=None, attention_mask=None):
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.values.append(value)
        self.log_probs.append(log_prob)
        self.dones.append(done)
        if input_ids is not None:
            self.input_ids.append(input_ids.cpu())
        if attention_mask is not None:
            self.attention_masks.append(attention_mask.cpu())

    def clear(self):
        for attr in [self.states, self.actions, self.rewards, self.values,
                     self.log_probs, self.dones, self.input_ids,
                     self.attention_masks, self.tp_pcts, self.sl_pcts]:
            attr.clear()

    def __len__(self):
        return len(self.states)


class PPOTrainer:
    """PPO training for equity FlagTraderModel."""

    def __init__(
        self,
        model: FlagTraderModel,
        prompt_builder: EquityPromptBuilder,
        lr: float = 3e-5,
        gamma: float = 0.99,
        gae_lambda: float = 0.95,
        clip_range: float = 0.2,
        ppo_epochs: int = 4,
        value_loss_coef: float = 0.5,
        entropy_coef: float = 0.01,
        max_grad_norm: float = 0.5,
    ) -> None:
        self.model = model
        self.prompt_builder = prompt_builder
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_range = clip_range
        self.ppo_epochs = ppo_epochs
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm
        self.optimizer = torch.optim.AdamW(model.get_trainable_params(), lr=lr)
        self.buffer = RolloutBuffer()

    def _obs_to_prompt(self, obs: dict[str, np.ndarray]) -> str:
        candles, portfolio, history = obs_to_prompt_inputs_equity(obs)
        return self.prompt_builder.build_prompt(candles, portfolio, history)

    @torch.no_grad()
    def collect_rollout_multi_asset(
        self,
        envs: dict[str, EquityTradingEnv],
        num_steps: int = 250,
        steps_per_block: int = 25,
    ) -> dict[str, float]:
        """Collect rollout rotating across multiple equity environments in blocks."""
        self.model.eval()
        self.buffer.clear()

        env_names = list(envs.keys())
        rng = np.random.default_rng()
        rng.shuffle(env_names)

        obs_map = {}
        episode_return_map = {}
        for name, env in envs.items():
            obs_map[name], _ = env.reset()
            episode_return_map[name] = 0.0

        episode_returns = []
        all_rewards = []
        assets_used = set()
        total_collected = 0
        asset_idx = 0

        while total_collected < num_steps:
            name = env_names[asset_idx % len(env_names)]
            asset_idx += 1
            env = envs[name]
            assets_used.add(name)

            block_steps = min(steps_per_block, num_steps - total_collected)
            for _ in range(block_steps):
                obs = obs_map[name]
                prompt = self._obs_to_prompt(obs)
                result = self.model.get_action(prompt, return_tokens=True)
                action, value, log_prob, tp_pct, sl_pct, input_ids, attention_mask = result

                env.set_tp_sl(tp_pct, sl_pct)
                next_obs, reward, terminated, truncated, info = env.step(action)
                done = terminated or truncated

                self.buffer.add(
                    prompt, action, reward, value, log_prob, done,
                    input_ids=input_ids, attention_mask=attention_mask,
                )
                self.buffer.tp_pcts.append(tp_pct)
                self.buffer.sl_pcts.append(sl_pct)
                episode_return_map[name] += reward
                all_rewards.append(reward)
                total_collected += 1

                if done:
                    episode_returns.append(episode_return_map[name])
                    episode_return_map[name] = 0.0
                    obs_map[name], _ = env.reset()
                else:
                    obs_map[name] = next_obs

        return {
            "mean_reward": float(np.mean(all_rewards)) if all_rewards else 0.0,
            "num_episodes_completed": len(episode_returns),
            "mean_episode_return": float(np.mean(episode_returns)) if episode_returns else 0.0,
            "num_assets_used": len(assets_used),
        }

    def compute_gae(self, rewards, values, dones, last_value=0.0):
        n = len(rewards)
        advantages = torch.zeros(n)
        last_gae = 0.0
        for t in reversed(range(n)):
            next_value = last_value if t == n - 1 else values[t + 1]
            next_non_terminal = 0.0 if dones[t] else 1.0
            delta = rewards[t] + self.gamma * next_value * next_non_terminal - values[t]
            last_gae = delta + self.gamma * self.gae_lambda * next_non_terminal * last_gae
            advantages[t] = last_gae
        returns = advantages + torch.tensor(values, dtype=torch.float32)
        return advantages, returns

    def _pad_and_batch_tokens(self):
        if self.buffer.input_ids:
            max_len = max(t.shape[1] for t in self.buffer.input_ids)
            pad_id = self.model.tokenizer.pad_token_id or 0
            padded_ids, padded_masks = [], []
            for ids, mask in zip(self.buffer.input_ids, self.buffer.attention_masks):
                seq_len = ids.shape[1]
                if seq_len < max_len:
                    pad_len = max_len - seq_len
                    ids = torch.cat([ids, torch.full((1, pad_len), pad_id, dtype=ids.dtype)], dim=1)
                    mask = torch.cat([mask, torch.zeros(1, pad_len, dtype=mask.dtype)], dim=1)
                padded_ids.append(ids)
                padded_masks.append(mask)
            return torch.cat(padded_ids, dim=0), torch.cat(padded_masks, dim=0)
        else:
            tokens = self.model.tokenizer(
                self.buffer.states, return_tensors="pt", max_length=512, truncation=True, padding=True,
            )
            return tokens["input_ids"], tokens["attention_mask"]

    def update(self, mini_batch_size: int = 4) -> dict[str, float]:
        """PPO update."""
        self.model.train()
        advantages, returns = self.compute_gae(
            self.buffer.rewards, self.buffer.values, self.buffer.dones,
        )
        if len(advantages) > 1:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        old_log_probs = torch.stack(self.buffer.log_probs).detach()
        actions_tensor = torch.tensor(self.buffer.actions, dtype=torch.long)
        all_input_ids, all_attention_masks = self._pad_and_batch_tokens()

        use_amp = self.model.device.type == "cuda"
        total_policy_loss = total_value_loss = total_entropy = total_kl = 0.0
        num_batches = 0
        n = len(self.buffer)

        for _ in range(self.ppo_epochs):
            indices = torch.randperm(n)
            for start in range(0, n, mini_batch_size):
                end = min(start + mini_batch_size, n)
                batch_idx = indices[start:end]

                b_input_ids = all_input_ids[batch_idx].to(self.model.device)
                b_attention_mask = all_attention_masks[batch_idx].to(self.model.device)
                b_actions = actions_tensor[batch_idx].to(self.model.device)
                b_old_log_probs = old_log_probs[batch_idx].to(self.model.device)
                b_advantages = advantages[batch_idx].to(self.model.device)
                b_returns = returns[batch_idx].to(self.model.device)

                amp_ctx = torch.amp.autocast("cuda", dtype=torch.bfloat16) if use_amp else contextlib.nullcontext()
                with amp_ctx:
                    new_log_probs, new_values, entropy, pred_tp, pred_sl = self.model.run_assessment(
                        b_input_ids, b_attention_mask, b_actions
                    )
                    ratio = torch.exp(new_log_probs - b_old_log_probs)
                    surr1 = ratio * b_advantages
                    surr2 = torch.clamp(ratio, 1.0 - self.clip_range, 1.0 + self.clip_range) * b_advantages
                    policy_loss = -torch.min(surr1, surr2).mean()
                    value_loss = nn.functional.mse_loss(new_values, b_returns)

                    batch_tp = torch.tensor(self.buffer.tp_pcts, dtype=torch.float32, device=self.model.device)[batch_idx]
                    batch_sl = torch.tensor(self.buffer.sl_pcts, dtype=torch.float32, device=self.model.device)[batch_idx]
                    tp_loss = (b_advantages.detach().abs() * (pred_tp - batch_tp).pow(2)).mean()
                    sl_loss = (b_advantages.detach().abs() * (pred_sl - batch_sl).pow(2)).mean()

                    loss = (
                        policy_loss
                        + self.value_loss_coef * value_loss
                        - self.entropy_coef * entropy.mean()
                        + 0.1 * (tp_loss + sl_loss)
                    )

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.get_trainable_params(), self.max_grad_norm)
                self.optimizer.step()

                total_policy_loss += policy_loss.item()
                total_value_loss += value_loss.item()
                total_entropy += entropy.mean().item()
                with torch.no_grad():
                    total_kl += (b_old_log_probs - new_log_probs).mean().item()
                num_batches += 1

        self.buffer.clear()
        denom = max(num_batches, 1)
        return {
            "policy_loss": total_policy_loss / denom,
            "value_loss": total_value_loss / denom,
            "entropy": total_entropy / denom,
            "approx_kl": total_kl / denom,
        }

In [ ]:
# ============================================================
# Create environments from downloaded data
# Use first 18 months for training, last 6 for testing
# ============================================================

WINDOW_SIZE = 20
TRAIN_RATIO = 0.75  # ~18 months out of 24

train_envs = {}
test_envs = {}

for symbol, df in all_candle_data.items():
    if len(df) < 100:  # Need enough data
        continue

    candles_np = df[["open", "high", "low", "close", "volume"]].values.astype(np.float32)
    split_idx = int(len(candles_np) * TRAIN_RATIO)

    train_candles = candles_np[:split_idx]
    test_candles = candles_np[split_idx:]

    if len(train_candles) > WINDOW_SIZE + 10:
        train_envs[symbol] = EquityTradingEnv(
            candles=train_candles,
            initial_cash=10_000.0,
            transaction_cost_bps=1.0,
            window_size=WINDOW_SIZE,
            symbol=symbol,
        )

    if len(test_candles) > WINDOW_SIZE + 10:
        test_envs[symbol] = EquityTradingEnv(
            candles=test_candles,
            initial_cash=10_000.0,
            transaction_cost_bps=1.0,
            window_size=WINDOW_SIZE,
            symbol=symbol,
        )

print(f"Training environments: {len(train_envs)}")
print(f"Test environments: {len(test_envs)}")

In [ ]:
# ============================================================
# Training loop
# 500 updates x 250 steps, checkpoint every 50
# ============================================================

TOTAL_UPDATES = 500
STEPS_PER_ROLLOUT = 250
STEPS_PER_BLOCK = 25  # consecutive steps on same asset before rotating
SAVE_EVERY = 50
LOG_EVERY = 10

SAVE_DIR = Path("flag_trader_equity")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

prompt_builder = EquityPromptBuilder(candle_window=WINDOW_SIZE)
trainer = PPOTrainer(
    model=model,
    prompt_builder=prompt_builder,
    lr=3e-5,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ppo_epochs=4,
    value_loss_coef=0.5,
    entropy_coef=0.01,
    max_grad_norm=0.5,
)

all_stats = []

for update_idx in range(1, TOTAL_UPDATES + 1):
    # Collect rollout across multiple assets
    rollout_stats = trainer.collect_rollout_multi_asset(
        train_envs, num_steps=STEPS_PER_ROLLOUT, steps_per_block=STEPS_PER_BLOCK,
    )

    # PPO update
    update_stats = trainer.update(mini_batch_size=4)

    stats = {**rollout_stats, **update_stats, "update": update_idx}
    all_stats.append(stats)

    # Log
    if update_idx % LOG_EVERY == 0:
        print(
            f"Update {update_idx}/{TOTAL_UPDATES} | "
            f"reward: {stats['mean_reward']:.4f} | "
            f"policy_loss: {stats['policy_loss']:.4f} | "
            f"value_loss: {stats['value_loss']:.4f} | "
            f"assets: {stats.get('num_assets_used', 0)}"
        )

    # Save checkpoint
    if update_idx % SAVE_EVERY == 0:
        ckpt_path = SAVE_DIR / f"checkpoint_{update_idx}.pt"
        model.save_trainable(ckpt_path)
        print(f"  Saved checkpoint: {ckpt_path}")

# Save final
final_path = SAVE_DIR / "final_model.pt"
model.save_trainable(final_path)
print(f"\nTraining complete. Final model saved to: {final_path}")

## 7. Walk-Forward Testing

Train su primi 18 mesi, test su ultimi 6.
Metriche: Sharpe, win rate, profit factor, max drawdown.
Plot equity curve.

In [ ]:
# ============================================================
# Test on out-of-sample (last 6 months)
# ============================================================

@torch.no_grad()
def run_backtest_on_symbol(
    model: FlagTraderModel,
    prompt_builder: EquityPromptBuilder,
    env: EquityTradingEnv,
    max_steps: int = 500,
) -> dict:
    """Run full backtest on a single symbol env."""
    model.eval()
    obs, _ = env.reset()
    total_values = [env.initial_cash]
    actions_taken = []
    trades = []  # list of realized pnl values

    for step in range(max_steps):
        candles, portfolio, history = obs_to_prompt_inputs_equity(obs)
        prompt = prompt_builder.build_prompt(candles, portfolio, history)
        action, _, _, tp_pct, sl_pct = model.get_action(prompt)
        env.set_tp_sl(tp_pct, sl_pct)
        obs, reward, terminated, truncated, info = env.step(action)

        total_values.append(info["total_value"])
        actions_taken.append(info["executed_action"])
        if info["step_pnl"] != 0:
            trades.append(info["step_pnl"])

        if terminated or truncated:
            break

    # Compute metrics
    values = np.array(total_values)
    returns = np.diff(values) / values[:-1]

    # Sharpe (annualized, 252 trading days)
    if len(returns) > 1 and np.std(returns) > 1e-12:
        sharpe = (np.mean(returns) / np.std(returns)) * np.sqrt(252)
    else:
        sharpe = 0.0

    # Win rate
    if trades:
        wins = sum(1 for t in trades if t > 0)
        win_rate = wins / len(trades) * 100
    else:
        win_rate = 0.0

    # Profit factor
    gross_profit = sum(t for t in trades if t > 0)
    gross_loss = abs(sum(t for t in trades if t < 0))
    pf = gross_profit / gross_loss if gross_loss > 0 else float("inf") if gross_profit > 0 else 0.0

    # Max drawdown
    peak = values[0]
    max_dd = 0.0
    for v in values:
        peak = max(peak, v)
        dd = (peak - v) / peak * 100
        max_dd = max(max_dd, dd)

    # Action distribution
    action_counts = [0, 0, 0]
    for a in actions_taken:
        action_counts[a] += 1
    total_a = sum(action_counts) or 1

    return {
        "total_return_pct": (values[-1] / values[0] - 1) * 100,
        "sharpe": sharpe,
        "win_rate": win_rate,
        "profit_factor": pf,
        "max_drawdown_pct": max_dd,
        "num_trades": len(trades),
        "actions_dist": [c / total_a for c in action_counts],
        "equity_curve": values,
    }

In [ ]:
# Run backtest on a sample of test symbols
test_symbols = list(test_envs.keys())[:20]  # Sample 20 for speed

all_results = {}
for symbol in tqdm(test_symbols, desc="Backtesting"):
    env = test_envs[symbol]
    result = run_backtest_on_symbol(model, prompt_builder, env)
    all_results[symbol] = result

# Aggregate metrics
returns = [r["total_return_pct"] for r in all_results.values()]
sharpes = [r["sharpe"] for r in all_results.values()]
win_rates = [r["win_rate"] for r in all_results.values() if r["num_trades"] > 0]
pfs = [r["profit_factor"] for r in all_results.values() if r["num_trades"] > 0 and r["profit_factor"] < 100]
drawdowns = [r["max_drawdown_pct"] for r in all_results.values()]

print("=" * 60)
print("WALK-FORWARD RESULTS (Out-of-Sample, last 6 months)")
print("=" * 60)
print(f"Symbols tested: {len(all_results)}")
print(f"Mean return:       {np.mean(returns):.2f}% (median: {np.median(returns):.2f}%)")
print(f"Mean Sharpe:       {np.mean(sharpes):.2f} (median: {np.median(sharpes):.2f})")
print(f"Mean win rate:     {np.mean(win_rates):.1f}%" if win_rates else "No trades")
print(f"Mean PF:           {np.mean(pfs):.2f}" if pfs else "No trades")
print(f"Mean max DD:       {np.mean(drawdowns):.2f}%")
print()

# Per-symbol breakdown
print(f"{'Symbol':<8} {'Return%':>8} {'Sharpe':>8} {'WR%':>6} {'PF':>6} {'MaxDD%':>7} {'Trades':>7}")
print("-" * 56)
for sym in sorted(all_results.keys()):
    r = all_results[sym]
    pf_str = f"{r['profit_factor']:.2f}" if r['profit_factor'] < 100 else "inf"
    print(
        f"{sym:<8} {r['total_return_pct']:>7.2f}% {r['sharpe']:>7.2f} "
        f"{r['win_rate']:>5.1f}% {pf_str:>6} {r['max_drawdown_pct']:>6.2f}% {r['num_trades']:>6}"
    )

In [ ]:
# Plot equity curves for top and bottom performers
sorted_by_return = sorted(all_results.items(), key=lambda x: x[1]["total_return_pct"], reverse=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Equity Curves - Walk-Forward Test (OOS)", fontsize=14)

# Top 2
for idx, (sym, result) in enumerate(sorted_by_return[:2]):
    ax = axes[0, idx]
    ax.plot(result["equity_curve"], linewidth=1.2)
    ax.set_title(f"{sym} - Return: {result['total_return_pct']:.1f}%, Sharpe: {result['sharpe']:.2f}")
    ax.set_xlabel("Steps")
    ax.set_ylabel("Portfolio Value ($)")
    ax.axhline(y=10_000, color="gray", linestyle="--", alpha=0.5)
    ax.grid(True, alpha=0.3)

# Bottom 2
for idx, (sym, result) in enumerate(sorted_by_return[-2:]):
    ax = axes[1, idx]
    ax.plot(result["equity_curve"], linewidth=1.2, color="red")
    ax.set_title(f"{sym} - Return: {result['total_return_pct']:.1f}%, Sharpe: {result['sharpe']:.2f}")
    ax.set_xlabel("Steps")
    ax.set_ylabel("Portfolio Value ($)")
    ax.axhline(y=10_000, color="gray", linestyle="--", alpha=0.5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("equity_curves_walkforward.png", dpi=150, bbox_inches="tight")
plt.show()

# Training loss curves
if all_stats:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    updates = [s["update"] for s in all_stats]

    axes[0].plot(updates, [s["mean_reward"] for s in all_stats], alpha=0.7)
    axes[0].set_title("Mean Reward per Update")
    axes[0].set_xlabel("Update")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(updates, [s["policy_loss"] for s in all_stats], alpha=0.7)
    axes[1].set_title("Policy Loss")
    axes[1].set_xlabel("Update")
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(updates, [s["value_loss"] for s in all_stats], alpha=0.7)
    axes[2].set_title("Value Loss")
    axes[2].set_xlabel("Update")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

## 8. Save & Download

Salva il checkpoint finale e scaricalo da Colab.

In [ ]:
# Final checkpoint is already saved at flag_trader_equity/final_model.pt
print(f"Final model: {final_path}")
print(f"File size: {final_path.stat().st_size / 1024 / 1024:.1f} MB")

# Save training stats
stats_path = SAVE_DIR / "training_stats.json"
with open(stats_path, "w") as f:
    json.dump(all_stats, f, indent=2)
print(f"Training stats: {stats_path}")

# Save backtest results
results_path = SAVE_DIR / "backtest_results.json"
results_serializable = {
    sym: {k: v for k, v in res.items() if k != "equity_curve"}
    for sym, res in all_results.items()
}
with open(results_path, "w") as f:
    json.dump(results_serializable, f, indent=2)
print(f"Backtest results: {results_path}")

# List all saved files
print(f"\nAll files in {SAVE_DIR}:")
for p in sorted(SAVE_DIR.iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name} ({size_mb:.1f} MB)")

In [ ]:
# Download from Colab
try:
    from google.colab import files
    # Download the final model
    files.download(str(final_path))
    # Download backtest results
    files.download(str(results_path))
    # Download plots
    files.download("equity_curves_walkforward.png")
    files.download("training_curves.png")
    print("Downloads started!")
except ImportError:
    print("Not running on Colab. Files saved locally:")
    print(f"  {final_path}")
    print(f"  {results_path}")